# Perception Flow & Evaluations

## Utilities

In [1]:
# Get the root path and data paths
#

from pathlib import Path


def repo_root(marker: str = "uv.lock") -> Path:
    """Nearest ancestor of the working directory containing *marker*."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / marker).is_file():
            return candidate
    raise FileNotFoundError(f"No {marker} found above {start}")

DATA_IN = repo_root() / "data_in"



# Perception Flow For Benchmarked Text

In [ ]:
# Get benchmark text, with intended affective states
# BRIGHTER Benchmark Dataset

from dataclasses import asdict
from functools import partial

import pandas as pd
from tqdm.auto import tqdm

from asa.core.affect import AffectVector, Utterance
from asa.core.representations import BASIC4, EKMAN6
from asa.perception.decode_keyword import BASIC4_KEYWORDS, EKMAN6_KEYWORDS, KeywordDecoder

# Get the raw benchmark
benchmark_raw_df = pd.read_parquet(DATA_IN / "brighter_emotions.parquet")

# Benchmark Differences
BENCHMARK_TO_EKMAN6 = {"joy": "happiness"}

# Representation Details
REP = EKMAN6        
KEYWORDS = EKMAN6_KEYWORDS 
ALIASES = BENCHMARK_TO_EKMAN6
# BLANK_VALUES = {axis_key(a): REP.rest for a in REP.axes}
NO_AFFECT = "(none)"     
AXES = [str(a) for a in REP.axes]

# A local helper function for mapping to an AffectVector
def to_affect_vector(labels, rep=EKMAN6, aliases=BENCHMARK_TO_EKMAN6):
    """Local helper function - Map to AffectVector object. None where no labels are recorded."""
    # if labels is None:
    #     return None
    # if isinstance(labels, str):
    #     labels = [labels]    

    values = {axis: rep.rest for axis in rep.axes}
    hi = rep.value_range[1] 

    for label in labels:
        axis = str(label).strip().lower()
        axis = aliases.get(axis, axis)
        if axis not in values:
            raise ValueError(f"{axis!r} is not an axis of {rep.axes}")
        values[axis] = hi
    affect = AffectVector(representation=rep.id,
                         values=values)
    return affect

# Convert raw benchmark to ID, text, AffectVector
benchmark_df = pd.DataFrame({"id": benchmark_raw_df["id"].astype("string"),
                     "text": benchmark_raw_df["text"],
                     "intended_affect": benchmark_raw_df["emotions"].map(partial(to_affect_vector, rep=REP, aliases=ALIASES))})


# Decode the benchmark text
results, errors = [], []
emotion_decoder = KeywordDecoder(representation=REP, table=KEYWORDS)

for row in tqdm(benchmark_df.to_dict("records")):
    utterance = Utterance(text=row["text"], 
                          source="input:benchmark", 
                          intended=row["intended_affect"])
    try:
        observation = await emotion_decoder.decode(utterance)
    except Exception as e:
        errors.append({"id": row["id"], "error": repr(e)})        
        continue

    results.append({"id": row["id"],
                    "utterance": asdict(utterance),
                    "observation": asdict(observation),
                    })

# Collate the Evaluation DF
def top_axis(values):
    """Dominant axis, or None where every axis is at rest — ie no affect expressed."""
    if all(v == REP.rest for v in values.values()):
        return NO_AFFECT
    return max(values, key=values.get)

eval_df = pd.DataFrame([{"id": r["id"],
                         "text": r["utterance"]["text"],
                         "intended_top": top_axis(r["utterance"]["intended"]["values"]),
                         "decoded_top": top_axis(r["observation"]["affect"]["values"]),
                        "rationale": r["observation"]["rationale"],
                         "confidence": r["observation"]["confidence"],
                         }
                        for r in results])


  0%|          | 0/2764 [00:00<?, ?it/s]

In [8]:
def fired_axes(values, rep=REP):
    """Every axis above rest — the multi-label view, as opposed to top_axis."""
    return {str(axis) for axis, v in values.items() if v > rep.rest}

In [9]:
eval_df = pd.DataFrame([{"id": r["id"],
                         "text": r["utterance"]["text"],
                         "intended_set": sorted(fired_axes(r["utterance"]["intended"]["values"])),
                         "decoded_set": sorted(fired_axes(r["observation"]["affect"]["values"])),
                         "rationale": r["observation"]["rationale"],
                         "confidence": r["observation"]["confidence"],
                         }
                        for r in results])

In [10]:
import numpy as np

Y_true = np.array([[a in s for a in AXES] for s in eval_df["intended_set"]])
Y_pred = np.array([[a in s for a in AXES] for s in eval_df["decoded_set"]])

per_axis = pd.DataFrame({"axis": AXES,
                         "n_true": Y_true.sum(0),
                         "tp": (Y_true & Y_pred).sum(0),
                         "fn": (Y_true & ~Y_pred).sum(0),
                         "fp": (~Y_true & Y_pred).sum(0),
                         })
per_axis["recall"] = (per_axis.tp / (per_axis.tp + per_axis.fn)).round(3)
per_axis["precision"] = (per_axis.tp / (per_axis.tp + per_axis.fp)).round(3)
per_axis

,axis,n_true,tp,fn,fp,recall,precision
0,anger,333,3,330,1,0.009,0.750
1,disgust,0,0,0,1,NaN,0.000
2,fear,1610,30,1580,1,0.019,0.968
3,happiness,674,41,633,35,0.061,0.539
4,sadness,876,22,854,0,0.025,1.000
5,surprise,839,6,833,0,0.007,1.000


In [17]:
from sklearn.preprocessing import MultiLabelBinarizer

AXES = [str(a) for a in REP.axes]

mlb = MultiLabelBinarizer(classes=AXES)
Y_true = mlb.fit_transform(eval_df["intended_set"])
Y_pred = mlb.transform(eval_df["decoded_set"])

In [18]:
from sklearn.metrics import classification_report

print(classification_report(Y_true, Y_pred, target_names=AXES, zero_division=0))

              precision    recall  f1-score   support

       anger       0.75      0.01      0.02       333
     disgust       0.00      0.00      0.00         0
        fear       0.97      0.02      0.04      1610
   happiness       0.54      0.06      0.11       674
     sadness       1.00      0.03      0.05       876
    surprise       1.00      0.01      0.01       839

   micro avg       0.73      0.02      0.05      4332
   macro avg       0.71      0.02      0.04      4332
weighted avg       0.90      0.02      0.04      4332
 samples avg       0.04      0.03      0.03      4332



In [15]:
from sklearn.metrics import multilabel_confusion_matrix

mcm = multilabel_confusion_matrix(Y_true, Y_pred)

per_axis = pd.DataFrame({"axis": AXES,
                         "n_true": Y_true.sum(0),
                         "tp": mcm[:, 1, 1],
                         "fn": mcm[:, 1, 0],
                         "fp": mcm[:, 0, 1],
                         "tn": mcm[:, 0, 0],
                         })
per_axis["recall"] = (per_axis.tp / per_axis.n_true).round(3)
per_axis["precision"] = (per_axis.tp / (per_axis.tp + per_axis.fp)).round(3)
per_axis

,axis,n_true,tp,fn,fp,tn,recall,precision
0,anger,333,3,330,1,2430,0.009,0.750
1,disgust,0,0,0,1,2763,NaN,0.000
2,fear,1610,30,1580,1,1153,0.019,0.968
3,happiness,674,41,633,35,2055,0.061,0.539
4,sadness,876,22,854,0,1888,0.025,1.000
5,surprise,839,6,833,0,1925,0.007,1.000


In [16]:
overlap = pd.DataFrame(Y_true.T @ Y_pred, index=AXES, columns=AXES)
overlap[NO_AFFECT] = ((Y_pred.sum(1) == 0)[:, None] * Y_true).sum(0)   # intended, nothing fired
overlap

,anger,disgust,fear,happiness,sadness,surprise,(none)
anger,3,1,0,7,2,0,320
disgust,0,0,0,0,0,0,0
fear,2,1,30,26,10,2,1539
happiness,1,0,1,41,1,2,628
sadness,1,0,5,18,22,0,831
surprise,0,0,9,12,2,6,810


In [3]:
scored = eval_df.dropna(subset=["intended_top"])
print(f"scoring {len(scored)} of {len(eval_df)} rows; {len(eval_df) - len(scored)} lack ground truth")

scoring 2764 of 2764 rows; 0 lack ground truth


In [4]:
print(f"benchmark rows : {len(benchmark_df)}")
print(f"decoded ok     : {len(results)}")
print(f"errors         : {len(errors)}")
print(f"in eval_df     : {len(eval_df)}")

benchmark rows : 2764
decoded ok     : 2764
errors         : 0
in eval_df     : 2764


In [ ]:
flat_results = pd.json_normalize(results)

## Runs

In [ ]:
# Get the test text and ground-truth affect
#

import pandas as pd

data_file = Path(DATA_IN / "benchmark_text.csv")
df_benchmark = pd.read_csv(data_file,
                           dtype={"text": "string", "basic4/1": "category", "ekman6/1": "category"})

df_benchmark.info() 
display(df_benchmark.describe())
display(df_benchmark.sample(5))


In [ ]:
# Decode the sentences
#

from dataclasses import asdict

from asa.core.affect import AffectVector, Utterance  # noqa: F811
from asa.core.representations import BASIC4, EKMAN6  # noqa: F811
from asa.perception.decode_keyword import BASIC4_KEYWORDS, EKMAN6_KEYWORDS, KeywordDecoder  # noqa: F811

basic4_decoder = KeywordDecoder(representation=BASIC4, table=BASIC4_KEYWORDS)
label_col = BASIC4.id
blank_values = {axis: BASIC4.rest for axis in BASIC4.axes}

# observations = []
results = []
for row in df_benchmark.to_dict("records"):
    print(row)
    label = row[label_col]
    # intended_affect = None
    # if not pd.isna(label):
    #     intended_affect = AffectVector(representation=BASIC4.id, values= blank_values | {label: 1.0})
    values = blank_values if pd.isna(label) else blank_values | {label: 1.0}
    intended_affect = AffectVector(representation=BASIC4.id, values=values)
    utterance = Utterance(text=row["text"], source="input:benchmark", intended=intended_affect)
    print(utterance)
    affect_observation = await basic4_decoder.decode(utterance)
    print(affect_observation)
    # observations.append(affect_observation)

    results.append({"utterance": asdict(utterance), "observation": asdict(affect_observation)})

flat_results = pd.json_normalize(results)
# flat_df = pd.json_normalize([asdict(o) for o in observations])